# AI-Powered HR Assistant using GPT-3.5 Turbo and ChromaDB

## Objective
This project builds a conversational AI assistant that answers queries based on 
Nestlé’s HR Policy document.

The system uses:
- OpenAI Embeddings for vector representation
- ChromaDB for similarity search
- GPT-3.5 Turbo for answer generation
- Gradio for user interface

The system follows a Retrieval-Augmented Generation (RAG) architecture to:
- Prevent hallucinations
- Ensure answers are grounded in policy
- Provide page number references
- Display similarity scores

## 1. Introduction

This project develops an AI-powered HR assistant capable of answering 
employee-related queries using Nestlé’s HR policy document.

The system uses a Retrieval-Augmented Generation (RAG) architecture, which 
combines semantic search with a language model to generate accurate and 
context-aware answers.

Unlike a standard chatbot, this assistant:
- Retrieves relevant policy text using vector embeddings
- Grounds responses strictly in document context
- Displays source page numbers
- Shows similarity scores
- Logs interactions for traceability

This approach prevents hallucination and ensures reliable policy-based responses.

# Install Required Libraries

In [1]:
!pip install openai chromadb pypdf gradio numpy

Defaulting to user installation because normal site-packages is not writeable
  Using cached chromadb-1.5.1-cp39-abi3-win_amd64.whl.metadata (7.3 kB)
  Using cached gradio-6.6.0-py3-none-any.whl.metadata (16 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.39.1-py3-none-any.whl.metadata (2.5 kB)
  Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached kubernetes-35.0.0-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached fastapi-0.131.0-py3-none-any.whl.metadata (30 kB)
  Using cached gradio_client-2.1.0-py3-none-any.whl.metadata (7.1 kB)
Using cached chromadb-1.5.1-cp39-abi3-win_amd64.whl (21.9 MB)
Using cached gradio-6.6.0-py3-none-any.whl (24.2 MB)
Using cached gradio_client-2.1.0-py3-none-any.whl (55 kB)
Using cached fastapi-0.131.0-py3-none-any.whl (103 kB)
Using cached kubernetes-35.0.0-py2.py3-none-any.whl (2.0 MB)
Using cached opentelemetry_exporter_otlp_proto_grpc-1.39.1-py3-none-any.whl (19 kB)
Using cached opentelemetry_sdk-1.39.1


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Import Libraries

In [4]:
import os
import json
import numpy as np
import chromadb
from datetime import datetime
from pypdf import PdfReader
import gradio as gr
from openai import OpenAI
print("Imported libraries Successfully")

Imported libraries Successfully


# Load API Key Securely

In [6]:
with open("config.json", "r") as f:
    config = json.load(f)

api_key = config["openai"]["api_key"]

client = OpenAI(api_key=api_key)

print("API Key Loaded Successfully")

API Key Loaded Successfully


# Load Nestlé HR Policy PDF

In [7]:
pdf_path = "the_nestle_hr_policy_pdf_2012.pdf"

reader = PdfReader(pdf_path)

pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    pages.append({
        "page_number": i + 1,
        "text": text
    })

print(f"Total Pages Loaded: {len(pages)}")

Total Pages Loaded: 8


# Chunking the Text

In [8]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

documents = []
chunk_id = 0

for page in pages:
    chunks = chunk_text(page["text"])
    for chunk in chunks:
        documents.append({
            "id": f"chunk_{chunk_id}",
            "text": chunk,
            "page_number": page["page_number"]
        })
        chunk_id += 1

print(f"Total Chunks Created: {len(documents)}")

Total Chunks Created: 26


# Generate Embeddings (Manual)

In [9]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

print("Generating embeddings...")

for doc in documents:
    doc["embedding"] = get_embedding(doc["text"])

print("Embeddings Generated Successfully")

Generating embeddings...
Embeddings Generated Successfully


## 2. Why Embeddings Are Used

Traditional keyword search fails when queries are phrased differently from 
document wording. To solve this, we convert document chunks into numerical 
vector representations (embeddings).

Embeddings allow semantic similarity comparison. This means:

- "learning policy"
- "training rules"
- "employee development guidelines"

can retrieve similar content even if exact keywords differ.

Cosine distance is used to measure similarity between vectors.
Lower distance means higher similarity.

To convert cosine distance into similarity score:
Similarity = 1 - (distance / 2)

This normalizes similarity into a 0–1 range.

# Setup ChromaDB

In [10]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="nestle_hr_policy")

for doc in documents:
    collection.add(
        ids=[doc["id"]],
        embeddings=[doc["embedding"]],
        documents=[doc["text"]],
        metadatas=[{"page_number": doc["page_number"]}]
    )

print("Documents stored in ChromaDB")

Documents stored in ChromaDB


# Retrieval Function

In [15]:
def retrieve_context(query, top_k=3):
    query_embedding = get_embedding(query)
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    
    retrieved_docs = results["documents"][0]
    retrieved_metadata = results["metadatas"][0]
    distances = results["distances"][0]
    
    similarity_scores = [round(1 - (d/2), 3) for d in distances]
    
    return retrieved_docs, retrieved_metadata, similarity_scores

## 3. Retrieval Strategy

When a user submits a query:

1. The query is converted into an embedding.
2. ChromaDB performs similarity search.
3. The top 3 most relevant chunks are retrieved.
4. Page numbers and similarity scores are recorded.

Even if a query does not exist in the document,
the system retrieves the closest semantic matches.
The language model then determines whether the answer
is available in context.

This design ensures factual grounding.

# GPT Answer Function

In [16]:
def generate_answer(query, temperature=0.2):
    try:
        docs, metadata, scores = retrieve_context(query)
        
        context = "\n\n".join(docs)
        
        system_prompt = """
        You are an HR assistant for Nestlé.
        Answer strictly using the provided context.
        If the answer is not found, say:
        "Information not available in the policy document."
        """
        
        user_prompt = f"""
        Context:
        {context}
        
        Question:
        {query}
        """
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            temperature=temperature,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        
        answer = response.choices[0].message.content
        
        # Memory Logging
        log_entry = {
            "timestamp": str(datetime.now()),
            "question": query,
            "pages": [m["page_number"] for m in metadata],
            "similarity_scores": scores,
            "answer": answer
        }
        
        with open("interaction_log.json", "a") as f:
            f.write(json.dumps(log_entry) + "\n")
        
        # Format output
        source_info = "\n".join([
            f"Page {metadata[i]['page_number']} — Similarity: {scores[i]}"
            for i in range(len(metadata))
        ])
        
        final_output = f"""
        Answer:
        {answer}
        
        Sources Used:
        {source_info}
        """
        
        return final_output
    
    except Exception as e:
        return f"Error occurred: {str(e)}"

## 4. Prompt Engineering Strategy

A structured prompt is used to control model behavior:

System Instruction:
- Act as Nestlé HR assistant.
- Answer only using provided context.
- If answer not found, respond with:
  "Information not available in the policy document."

This prevents hallucination and enforces strict document adherence.

Temperature control allows adjustment of response creativity.
For policy-based systems, a low temperature (0.2) is recommended.

## 5. Transparency and Explainability

To improve trust and accountability, the system displays:

- Answer text
- Source page numbers
- Similarity scores

Additionally, each interaction is logged with:
- Timestamp
- User query
- Retrieved page numbers
- Similarity scores
- Generated answer

This provides traceability and reproducibility.

# Gradio Interface

In [19]:
interface = gr.Interface(
    fn=generate_answer,
    inputs=[
        gr.Textbox(label="Ask a question about Nestlé HR Policy"),
        gr.Slider(0, 1, value=0.2, label="Temperature")
    ],
    outputs=gr.Textbox(label="Response"),
    title="Nestlé HR AI Assistant",
    description="AI-powered HR assistant with source page references and similarity scores."
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## 6. Evaluation and Testing

The system was tested with various queries:

Example 1: "learning policy"
- Retrieved relevant policy sections
- Returned structured HR learning framework
- Similarity scores above 0.4

Example 2: "performance evaluation"
- Correctly identified Performance Evaluation (PE) process
- Retrieved page 6 where performance tools are discussed

Example 3: "exit policy"
- No relevant content found
- System correctly responded:
  "Information not available in the policy document."

These tests confirm that:
- Retrieval works semantically
- Hallucinations are prevented
- Source transparency is maintained

## Deployment Instructions

1. Install dependencies:
   pip install openai chromadb pypdf gradio numpy

2. Run all notebook cells.

3. The Gradio interface will launch locally.

Optional:
- Use interface.launch(share=True) for public link.
- Deploy to Hugging Face Spaces using requirements.txt.

## 7. Conclusion

This project demonstrates the practical implementation of a Retrieval-Augmented 
Generation (RAG) system using GPT-3.5 Turbo and ChromaDB.

Key achievements:
- Manual embedding generation
- Vector similarity search
- Controlled prompt engineering
- Source transparency with similarity scores
- Interaction logging
- Gradio-based deployment interface

The system enhances HR operational efficiency by enabling quick and reliable 
access to policy information.

Future improvements may include:
- Similarity threshold filtering
- Role-based query handling
- Multi-document support
- Cloud deployment for enterprise use